# NEOARM: specimen images, optical response, and inversion status

GPU images support synthetic projector and condenser recovery under declared first-order optical models. A new 40k specimen image now uses the condenser's same shared-objective column, with 10.5 million Gaussian beamlets and image-derived fitting. However, an exact uniform-body ray sensitivity test finds large nonparaxial corrections in some condenser settings. **Whole-column recovery remains unfinished**: statistical recovery of linearized synthetic data does not establish an instrument-valid acquisition. The sections below distinguish these results and their limitations.

This notebook separates the specimen simulation, numerical convergence, conditional recovery, and instrument requirements. Review mode loads saved GPU results; the optional switch recomputes a 150k specimen image on the GPU. Opening the notebook does not launch a fit or bootstrap.

The [imaging-first recipe investigation](NEOARM_IMAGING_RECIPE.md) separates the accepted specimen-image test, which still uses effective independent objective halves, from the new shared-field information test. The earlier coupled fit used an unsupported astigmatic-angle law, and the original full specimen-feature likelihood failed at truth; both remain historical evidence, not physical recovery claims. [NEOARM_STATUS.md](NEOARM_STATUS.md) gives the current handoff and [the research archive](archive/neoarm_full_column_research.ipynb) preserves the old investigation. Synthetic DAC values are not instrument commands.

## 1. Choose the image scale

The saved dataset covers 40k, 80k, 150k, 300k and 600k on a 256 × 256 detector with 55 µm pixels, at 200 kV. Changing the scale selects a different registered region of the **same specimen**.

Reviewing saved images does not require a GPU. Fresh Gaussian propagation does. No global optimization or bootstrap runs automatically.


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("JAX_ENABLE_X64", "true")

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/temgym_core").is_dir())
EXAMPLE = REPO / "examples/lens_inversion"
GENERATED = REPO / "examples/generated/neoarm_inversion"
OUTPUT = GENERATED / "review"
sys.path.insert(0, str(EXAMPLE))

MAGNIFICATION = 80_000
RERENDER_150K_ON_GPU = False
EXPORT_FIGURES = True
DOSE_ELECTRONS = 20_000_000
READ_NOISE_E = 0.6


In [ ]:
import json
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display
from matplotlib.patches import Polygon, Rectangle

from neoarm_workflow import (
    draw_detector_images,
    fit_status, frozen_sample_view, load_saved_acquisition,
    expected_detector_counts,
    nominal_sample_requests, read_figure_metadata, reproduce_sample_image_gpu,
)
from temgym_core.constants import energy2wavelength
from temgym_core.multiscale_sample import MAGICALSampleAtlas, select_sample_view
from temgym_core.neoarm_calibration import hard_edge_column_from_config
from temgym_core.plotting import export_ray_figure

jax.config.update("jax_enable_x64", True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
CONFIG_PATH = EXAMPLE / "neoarm_inversion_config.toml"
with CONFIG_PATH.open("rb") as stream:
    config = tomllib.load(stream)
column = hard_edge_column_from_config(config)


In [ ]:
imaging = load_saved_acquisition(GENERATED / "physical_checkpoint_native_imaging_v5_gpu", CONFIG_PATH)
diffraction = load_saved_acquisition(GENERATED / "physical_checkpoint_native_diffraction_v3_gpu", CONFIG_PATH)
nominals = nominal_sample_requests(imaging)
by_magnification = {int(r.target_magnification): r for r in nominals}
if MAGNIFICATION not in by_magnification:
    raise ValueError(f"Choose a saved magnification: {tuple(by_magnification)}")

selected = by_magnification[MAGNIFICATION]
parameters = np.asarray(imaging.metadata["parameters"])
detector = imaging.detector
dacs = np.asarray(selected.raw_dac_values)
print(f"Verified {len(imaging.requests)} imaging + {len(diffraction.requests)} diffraction arrays.")
print("Saved renderer:", imaging.metadata["evaluation_method"])
print("Saved GPU:", ", ".join(imaging.metadata["gpu_device_kinds"]))
print("Current review backend:", jax.default_backend())
print("Configuration matches the archive; historical checkpoints do not record a source-code hash.")


## 2. One specimen, several fields of view

The atlas stores log **field** amplitude and phase-equivalent pathlength. Cubic splines provide the values, gradients and Hessians used by the Gaussian interaction. The pose is separate from the raster.

The present specimen is a **synthetic test object**: nominal 10 nm / 13 nm layers, four registered stacks, a 40 nm perforation, and a periodic silicon-spacing tile. Its lattice contrast is a pair of sinusoidal phase gratings, not an atomistic scattering calculation. Before an instrument experiment we need the actual certificate, a surveyed region map, and a suitable contrast model.

The colored outlines below are preview footprints for the archived five-scale column. Each is drawn at the corresponding stage position in the common specimen coordinates. Section 4 separately shows the frozen footprints used for the new measured-geometry images.


In [ ]:
sample_config = config["sample"]
atlas = MAGICALSampleAtlas.from_config(
    config,
    region_shape_yx=tuple(sample_config["region_shape_yx"]),
    authoritative_multilayer_shape_yx=tuple(sample_config["authoritative_multilayer_shape_yx"]),
    global_support_margin_m=sample_config["atlas_support_margin_m"],
)
views = {}
for request in nominals:
    J = detector.affine_xy @ np.asarray(
        column.sample_position_jacobian(parameters, request.raw_dac_values))
    views[int(request.target_magnification)] = select_sample_view(
        atlas, transfer_jacobian_xy=J,
        detector_shape_yx=detector.shape_yx,
        detector_pixel_size_yx_m=detector.pixel_size_yx_m,
        gaussian_radius_min_m=3.4e-9,
        lens_wobble_margin_m=.15e-9, calibrated_input_margin_m=1.10e-9,
        gaussian_support_margin_m=.10e-9, pose_uncertainty_margin_m=.10e-9,
        interpolation_halo_margin_m=.05e-9, z=column.sample_z_m,
    )
print(atlas.certificate.version)
print("Preview selections are recomputed; saved images retain their original frozen crops.")


In [ ]:
level = atlas.pyramid.levels[0]
fig, ax = plt.subplots(figsize=(8, 7), layout="constrained")
image = ax.imshow(level.log_amplitude_yx, origin="lower",
                  extent=np.asarray(level.bounds_xyxy_m)*1e9, cmap="Greys")
for number, region in enumerate(atlas.regions.values(), start=1):
    x0, x1, y0, y1 = np.asarray(region.bounds_xyxy_m)*1e9
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, ec="0.5", lw=.8))
    ax.text((x0+x1)/2, (y0+y1)/2, str(number), ha="center", color="#D55E00")
half = np.asarray(detector.shape_yx[::-1]) * np.asarray(detector.pixel_size_yx_m[::-1]) / 2
corners = np.array([[-1,-1], [1,-1], [1,1], [-1,1]]) * half
for color, request in zip(plt.cm.viridis(np.linspace(.08, .85, 5)), nominals):
    mag = int(request.target_magnification)
    J = detector.affine_xy @ np.asarray(column.sample_position_jacobian(parameters, request.raw_dac_values))
    world = corners @ np.linalg.inv(J).T + np.asarray(atlas.pose_translation_xy_m)
    local = np.asarray(views[mag].world_to_local(jnp.asarray(world)))
    ax.add_patch(Polygon(local*1e9, fill=False, ec=color, lw=1.7, label=f"{mag/1000:.0f}k"))
ax.set(xlabel="specimen x [nm]", ylabel="specimen y [nm]", title="One registered synthetic specimen")
ax.legend(ncols=3, frameon=False)
fig.colorbar(image, ax=ax, label="log field amplitude", shrink=.7)
plt.show()
display(Markdown("\n".join(f"{i}. {name}" for i, name in enumerate(atlas.regions, 1))))


In [ ]:
rows = ["| Scale | Feature | Region | Cubic spacing (nm) | Projected spacing (px) | Sampling |",
        "|---|---|---|---:|---:|---|"]
for mag, view in views.items():
    s = view.selection
    passed = s.meets_feature_sampling and s.meets_resolution
    rows.append(f"| {mag/1000:.0f}k | {s.feature_family} | {s.selected_region_name or 'atlas'} | "
                f"{s.selected_spacing_m*1e9:.4f} | {s.projected_feature_pixels:.2f} | "
                f"{'pass' if passed else 'insufficient'} |")
display(Markdown("\n".join(rows)))


Sampling is only one test. At 600k the in-focus phase-only lattice has almost no intensity contrast in this model, even though its spacing is sampled adequately. The archived analysis excludes those real-space Fourier measurements and demonstrates signed defocus diversity near ±19.6 nm. Diffraction is another source of indexed scale information.

The next cell inspects a line through the selected *saved* crop. Explicit spline derivatives, not differentiation through wrapped phase, supply the gradient and curvature.


In [ ]:
saved_view = frozen_sample_view(atlas, imaging, selected, z_m=column.sample_z_m)
centre = np.asarray(atlas.pose_translation_xy_m)
sample_pixel = min(detector.pixel_size_yx_m) / MAGNIFICATION
line_x = centre[0] + np.linspace(-80, 80, 401) * sample_pixel
points = jnp.asarray(np.column_stack((line_x, np.full(line_x.size, centre[1]))))
jet = saved_view.action_jet(points)
offset_nm = (line_x-centre[0])*1e9
fig, axes = plt.subplots(2, 3, figsize=(11, 5), layout="constrained")
for row, (action, label, value_scale) in enumerate(
        ((jet.pathlength, "pathlength [pm]", 1e12), (jet.log_amplitude, "log field amplitude", 1.))):
    curves = (action.value*value_scale,
              action.gradient_xy[:, 0]*value_scale*1e-9,
              action.hessian_xy[:, 0, 0]*value_scale*1e-18)
    for ax, values, suffix in zip(axes[row], curves, ("", " / nm", " / nm²")):
        ax.plot(offset_nm, values, color="#0072B2", lw=1.4)
        ax.set(xlabel="sample x offset [nm]", ylabel=label+suffix)
axes[0, 0].set_title("Value")
axes[0, 1].set_title("First derivative")
axes[0, 2].set_title("Second derivative")
plt.show()


## 3. The detector images and the controls that produced them

These are physical Gaussian-rendered intensity arrays, with a new reproducible Poisson/read-noise realization. The count conversion uses the incident beam normalization, **not each image's sum**, so transmission differences are retained. A shared color scale permits intensity comparisons. Bars show specimen distances along the detector's horizontal direction, including rotation and anisotropy.

This is useful to the inversion: the specimen supplies known spatial distances, while position/tilt responses constrain different transfer-matrix terms. An image that merely looks plausible is not evidence of a correct or unique lens fit.


In [ ]:
fig, nominal_counts = draw_detector_images(
    imaging, nominals, column, dose_electrons=DOSE_ELECTRONS, read_noise_e=READ_NOISE_E)
if EXPORT_FIGURES:
    export_ray_figure(fig, OUTPUT / "nominal_sample_images", metadata={
        "kind": "synthetic exposures from saved Gaussian intensities",
        "manifest_sha256": imaging.checkpoint.manifest_sha256,
        "frame_ids": [r.frame_id for r in nominals],
        "dose_electrons": DOSE_ELECTRONS, "read_noise_e": READ_NOISE_E,
        "noise_seed": "51000 + frame_id", "calibrated_instrument": False,
    })
plt.show()


In [ ]:
selected_requests = tuple(
    r for r in imaging.requests
    if r.optical_state_label == selected.optical_state_label
    and r.dac_approach == "forward" and r.repeat_index == 0
    and r.measurement in {"perforation", "position_basis", "angle_basis", "sample"}
)
fig, selected_counts = draw_detector_images(
    imaging, selected_requests, column, dose_electrons=DOSE_ELECTRONS, read_noise_e=READ_NOISE_E)
plt.show()
rows = ["| Control | Synthetic coordinate |", "|---|---:|"]
rows += [f"| {name} | {value:.8f} |" for name, value in zip(column.channel_names, dacs)]
display(Markdown("\n".join(rows)))
print("Calibrated inputs: +5 nm position; +1 µrad tilt. Objective controls fixed; corrector off.")
print("These are unscaled synthetic coordinates, NOT instrument DAC instructions.")


### Optional: recompute the 150k image on the GPU

This is a compact, inspectable reproduction of the saved 150k experiment. It generates a calibrated Gaussian at the sample, multiplies it by the cubic specimen, fits 480,249 Gaussian beamlets, and propagates through the downstream hard-edge lenses. It does not infer the condenser from a sample-plane input.

The grid and sample crop are frozen to the saved experiment. The helper rejects a missing GPU instead of falling back to the CPU. Its comparison with the old render is a **reproduction check**. The following cell separately evaluates an independent near-conjugate grid reference and tests the detector discrepancy against the declared noise. Neither is a full inverse-fit acceptance test. Fresh arbitrary-magnification production runs remain in the research archive until that orchestration is extracted.


In [ ]:
if RERENDER_150K_ON_GPU:
    demonstration = load_saved_acquisition(
        GENERATED / "physical_checkpoint_150k_256px_693x693_v11_gpu", CONFIG_PATH)
    fresh_render, transmitted_field = reproduce_sample_image_gpu(
        demonstration, 27, atlas, column, voltage_v=config["beam"]["voltage_v"])
    old_image = demonstration.image(27)
    relative_error = np.linalg.norm(fresh_render.intensity_yx-old_image) / np.linalg.norm(old_image)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), layout="constrained")
    for ax, values, title in zip(axes, (old_image, fresh_render.intensity_yx,
                                       fresh_render.intensity_yx-old_image),
                                ("Saved Gaussian render", "Fresh GPU render", "Fresh − saved")):
        artist = ax.imshow(values, origin="lower", cmap="gray" if title != "Fresh − saved" else "RdBu_r")
        ax.set(title=title, xlabel="detector column", ylabel="detector row")
        fig.colorbar(artist, ax=ax, shrink=.7)
    plt.show()
    print(f"Relative image L2 difference: {relative_error:.3g}")
else:
    print("Fresh GPU render disabled. Set RERENDER_150K_ON_GPU=True to reproduce it.")


In [ ]:
if RERENDER_150K_ON_GPU:
    from temgym_core.neoarm_calibration import conjugate_gaussian_grid_reference
    view_150k = frozen_sample_view(
        atlas, demonstration, demonstration.request(27), z_m=column.sample_z_m)
    metadata_150k = demonstration.metadata
    matrix_150k = np.asarray(column.matrix(
        np.asarray(metadata_150k["parameters"]), demonstration.request(27).raw_dac_values,
        segment="sample_to_detector"))
    waist = metadata_150k["sample_illumination_radius_m"]
    grid_reference = conjugate_gaussian_grid_reference(
        view_150k, matrix_150k, demonstration.detector,
        incident_centre_xy_m=atlas.pose_translation_xy_m,
        incident_waist_xy_m=(waist, waist),
        wavelength_m=float(energy2wavelength(config["beam"]["voltage_v"])),
        sample_bandlimit_spacing_m=metadata_150k["sample_grid_step_m"])
    gaussian_counts = expected_detector_counts(
        fresh_render.intensity_yx, demonstration.detector, DOSE_ELECTRONS)
    grid_counts = expected_detector_counts(
        grid_reference.intensity_yx, demonstration.detector, DOSE_ELECTRONS)
    residual_sigma = (gaussian_counts-grid_counts) / np.sqrt(grid_counts+READ_NOISE_E**2)
    reference_rms = float(np.sqrt(np.mean(residual_sigma**2)))
    fig, ax = plt.subplots(figsize=(5, 4), layout="constrained")
    limit = max(float(np.max(np.abs(residual_sigma))), 1e-12)
    artist = ax.imshow(residual_sigma, origin="lower", cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set(title="Gaussian − independent grid reference", xlabel="detector column", ylabel="detector row")
    fig.colorbar(artist, ax=ax, label="noise standard deviations")
    plt.show()
    print(f"Independent detector check: {reference_rms:.6g} noise σ RMS (limit 0.2)")
    assert reference_rms < 0.2


## 4. Specimen-rendering convergence and the historical fit

The historical 150k refinement experiment uses 480,249 → 751,689 → 1,172,889 beamlets, with an independent near-conjugate grid reference. It also repeats a **one-parameter** IL1 fit while other optical parameters are held fixed. The later 40k experiment below refines to 10.5 million beamlets and jointly refits seven image-calibration parameters.

The broader ledger records 145 accepted rendering signatures. That supports the recorded forward comparisons; it does not demonstrate recovered parameters for the full column. In particular, a linearized Fisher-error calculation is not the plan's required two-refinement *rerender-and-refit* test for all identifiable parameters.


In [ ]:
refinement = read_figure_metadata(
    GENERATED / "neoarm_physical_150k_production_refinement_v2_gpu.metadata.json")
fit_report = read_figure_metadata(
    GENERATED / "neoarm_physical_identifiable_subspace_v2.metadata.json")
status = fit_status(fit_report)
rows = ["| Test | Recorded result |", "|---|---:|"]
rows += [
    f"| 150k independent-reference detector error | {refinement['independent_reference_detector_rms_in_noise_sigma']:.3g} noise σ RMS |",
    f"| Largest 150k independent-reference observable error | {max(refinement['independent_reference_observable_residuals_in_sigma'].values()):.3g} σ |",
    f"| Historical global optimizer converged starts | {status['converged_starts']} |",
    f"| Historical global training residual | {status['training_rms_sigma']:,.0f} σ RMS |",
    f"| Historical global held-out residual | {status['held_out_rms_sigma']:,.0f} σ RMS |",
    f"| Historical full-column fit accepted | {status['accepted_synthetic_fit']} |",
    f"| Instrument acceptance | {status['instrument_accepted']} |",
]
display(Markdown("\n".join(rows)))


In [ ]:
beam_counts = np.array([480249, 751689, 1172889])
q = np.asarray(refinement["q_estimates"])
q_sigma = np.asarray(refinement["q_standard_errors"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
axes[0].plot(beam_counts/1e6, refinement["detector_residual_rms_in_noise_sigma"], "o-", color="#0072B2")
axes[0].set(xlabel="beamlets [million]", ylabel="image difference [noise σ RMS]",
            title="150k image: difference from finest grid")
axes[1].plot(beam_counts/1e6, (q-q[-1])/q_sigma[-1], "o-", color="#D55E00")
axes[1].set(xlabel="beamlets [million]", ylabel="IL1 q shift [conditional σ]",
            title="One-parameter fit: difference from finest")
plt.show()
print("Historical refinement reports are displayed here; they were not rerun by these cells.")
print("A failed full-column fit is not used as a fitted ray trace.")


### Image-based scale calibration at 40k and 80k

These Gaussian images use the approximate diagram geometry, automatically selected crops of the same atlas, finite pixel-area integration, and a detector-blur halo. The frozen cubic object is fitted with magnification, rotation, specimen x/y position, dose, illumination width, and attenuation contrast released together. Detector geometry, specimen certificate, and beam centre are conditional calibration inputs. This is image calibration, not individual lens recovery; the objective still uses independent effective halves in this demonstration.

The saved finite-pixel fits pass both truth-image and seven-parameter bias checks. Earlier beam-count refinements provide a separate convergence test; complete fitting-range crop coverage and optical-parameter recovery remain unproved. Fresh rendering and fitting are explicit commands in [the imaging-recipe report](NEOARM_IMAGING_RECIPE.md), not actions triggered by opening this notebook.

In [ ]:
import hashlib
metrology_path = GENERATED / "imaging_recipe/measured_specimen.json"
metrology = json.loads(metrology_path.read_text())
pixel_path = GENERATED / "imaging_recipe/finite_pixel_metrology.json"
pixel_metrology = json.loads(pixel_path.read_text())
for report in (metrology, pixel_metrology):
    for path, checksum in report["source_hashes"].items():
        if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
            raise ValueError(f"Stale specimen-calibration source: {path}")
rows = ["| Magnification | Beamlets | Truth image (σ RMS) | Noisy fit (σ RMS) | Gate |",
        "|---|---:|---:|---:|---|"]
for row in pixel_metrology["rows"]:
    rows.append(f"| {row['magnification']/1000:.0f}k | {row['beam_count']:,} | "
                f"{row['noiseless']['data_residual_rms_sigma']:.5g} | "
                f"{row['noisy']['data_residual_rms_sigma']:.4f} | {row['passed']} |")
display(Markdown("\n".join(rows)))
print("Pixel-integrated image calibration; physical lens and hardware acceptance remain false.")


In [ ]:
calibration_images = []
for row in pixel_metrology["rows"]:
    archive = row["comparison_arrays"]
    array_path = REPO / archive["path"]
    input_path = REPO / row["input_artifact"]
    for path, checksum in ((array_path, archive["sha256"]), (input_path, row["input_artifact_sha256"])):
        if hashlib.sha256(path.read_bytes()).hexdigest() != checksum:
            raise ValueError(f"Saved specimen-image checksum mismatch: {path}")
    with np.load(array_path, allow_pickle=False) as saved:
        data, fitted, sigma = (saved[key] for key in
            ("noisy_counts_yx", "fitted_counts_yx", "noise_sigma_yx"))
    state = next(r for r in metrology["rows"] if r["magnification"] == row["magnification"])
    with np.load(input_path, allow_pickle=False) as saved:
        np.testing.assert_array_equal(saved["raw_dac_values"], state["controls"])
        np.testing.assert_array_equal(saved["transfer_matrix"], state["transfer_matrix"])
    calibration_images.append((row["magnification"], data, fitted, (data-fitted)/sigma))
print("Accepted fresh GPU images: 10.50 million beamlets at 40k; 2.42 million at 80k.")


In [ ]:
from temgym_core.multiscale_sample import _sample_projection_numpy
level = atlas.pyramid.levels[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True, layout="constrained")
for ax, row in zip(axes, metrology["rows"]):
    s = row["selection"]
    with np.load(REPO / row["selected_artifact"], allow_pickle=False) as saved:
        world = saved["footprint_specimen_xy_m"][[0, 2, 3, 1]]
    projection, _ = _sample_projection_numpy(atlas.pose_rotation_rad, s["sample_tilt_xy_rad"])
    local = (world - np.asarray(s["sample_translation_xy_m"])) @ np.linalg.inv(projection).T
    image = ax.imshow(level.log_amplitude_yx, origin="lower", cmap="Greys",
                      extent=np.asarray(level.bounds_xyxy_m)*1e9)
    for region in atlas.regions.values():
        x0, x1, y0, y1 = np.asarray(region.bounds_xyxy_m)*1e9
        chosen = region.name == s["selected_region_name"]
        ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False,
            ec="#009E73" if chosen else "0.65", lw=1.8 if chosen else .6,
            label="Selected region" if chosen else None))
    ax.add_patch(Polygon(local*1e9, fill=False, ec="#0072B2", lw=2, label="Detector footprint"))
    x0, x1, y0, y1 = np.asarray(row["sample_view_bounds_xyxy_m"])*1e9
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False,
                          ec="#D55E00", ls="--", lw=1.4, label="Frozen cubic domain"))
    region_label = (s["selected_region_name"] or "global atlas").replace("_", " ")
    ax.set(xlabel="atlas-local x [nm]", ylabel="atlas-local y [nm]", aspect="equal",
           title=f"{row['magnification']/1000:.0f}k · {region_label}\n"
                 f"{s['feature_family'].replace('_', ' ')} · level {s['level_index']} · {s['selected_spacing_m']*1e9:.3f} nm knots")
    ax.legend(frameon=True, facecolor="white", framealpha=.92, fontsize=8, loc="lower left")
fig.colorbar(image, ax=axes, label="log field amplitude", shrink=.7)
if EXPORT_FIGURES:
    export_ray_figure(fig, OUTPUT / "measured_specimen_footprints", metadata={"report_sha256": hashlib.sha256(metrology_path.read_bytes()).hexdigest()})
plt.show()


In [ ]:
if calibration_images:
    fig, axes = plt.subplots(2, 3, figsize=(11, 7), layout="constrained")
    maximum = max(np.max(values) for _, data, fitted, _ in calibration_images for values in (data, fitted))
    for panels, (mag, data, fitted, difference) in zip(axes, calibration_images):
        for col, (ax, values) in enumerate(zip(panels, (data, fitted, difference))):
            kwargs = dict(cmap="RdBu_r", vmin=-4, vmax=4) if col == 2 else dict(cmap="gray", vmin=0, vmax=maximum)
            artist = ax.imshow(values, origin="lower", extent=(-7.04, 7.04, -7.04, 7.04), **kwargs)
            ax.set(xlabel="detector x [mm]", ylabel="detector y [mm]" if col == 0 else "",
                   title=(f"Synthetic exposure · {mag/1000:.0f}k", "Cubic-object fit", "Data − fit")[col])
            if col < 2:
                ax.plot([-6, -6+50e-9*mag*1e3], [-5.9, -5.9], color="#E69F00", lw=3)
                ax.text(-6, -5.2, "50 nm specimen", color="#E69F00", fontsize=8)
            fig.colorbar(artist, ax=ax, shrink=.75, label="noise σ" if col == 2 else "electrons / pixel")
    fig.suptitle("Measured-diagram surrogate · pixel-integrated Gaussian images and fits", fontsize=13)
    if EXPORT_FIGURES:
        export_ray_figure(fig, OUTPUT / "finite_pixel_sample_fits", metadata={
            "scope": "conditional conjugate image metrology, not individual-lens recovery",
            "report_sha256": hashlib.sha256(pixel_path.read_bytes()).hexdigest(),
            "images": [row["comparison_arrays"] for row in pixel_metrology["rows"]],
            "detector_pixel_m": 55e-6, "shape_yx": [256, 256]})
    plt.show()


## 5. A lens fit from synthetic image-derived responses

The PL1 pilot generates reference and ±x, ±y, ±αx, and ±αy vacuum images, using calibrated 5 nm position and 1 mrad angle steps. Of 25 candidate lens settings, two produce unresolved subpixel beams and are excluded by the declared width gate; the remaining 23 settings give 207 noisy 256×256 images. Each image fits its own centroid, covariance, and dose, including pixel integration, blur, and clipping. Independent Gaussian-beam propagation checks the narrowest and broadest admitted states.

The baseline below uses a nineteen-coordinate relaxation, fitting sixteen upstream coefficients plus the length/gap ratio and excitation law. Its full image-derived covariance exposes weak geometry precision. The physical fit and redesigned acquisition follow immediately below; they enforce a round magnetic upstream map, with unknown specimen magnetic shear. The 2 nm × 1.5 nm pure probe and calibrated inputs remain conditional preparations, separate from condenser calibration.

The nearly identical reference and tilt images below reflect the weak tilt response of this relay, whereas the position input produces a visible displacement. The earlier coupled benchmark and its unsupported independent condenser-angle law remain in [the historical recovery report](NEOARM_RECOVERY.md).

In [ ]:
pl1_path = GENERATED / "imaging_recipe/last_lens_images.json"
pl1 = json.loads(pl1_path.read_text())
branches = json.loads((GENERATED / "imaging_recipe/last_lens_image_branches.json").read_text())
for report in (pl1, branches):
    for path, checksum in report["source_hashes"].items():
        if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
            raise ValueError(f"Stale PL1 result: {path}")
pl1_arrays = REPO / pl1["arrays_path"]
if hashlib.sha256(pl1_arrays.read_bytes()).hexdigest() != pl1["arrays_sha256"]:
    raise ValueError("PL1 image checksum mismatch.")
with np.load(pl1_arrays, allow_pickle=False) as saved:
    pl1_images = saved["noisy_counts"]
    pl1_q = saved["candidate_q"][saved["generated_candidate_indices"]]
fit = branches["recovery_diagnostic"]
rows = ["| Quantity | Fitted error (%) | Local uncertainty (%) |", "|---|---:|---:|"]
for label, error, uncertainty in zip(("PL1 length", "Detector gap"),
        fit["length_gap_errors_percent"], fit["length_gap_standard_errors_percent"]):
    rows.append(f"| {label} | {error:+.2f} | {uncertainty:.2f} |")
display(Markdown("\n".join(rows)))
print(f"{pl1['generated_frame_count']} frames; {pl1['incident_electrons_per_frame']:g} incident electrons/frame.")
print("Branch starts / converged:", branches["start_count"], branches["converged_start_count"])
print("Global uniqueness proved:", branches["global_branch_uniqueness_proved"])


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6), layout="constrained")
shown_states, shown_frames = (0, len(pl1_images)-1), (0, 1, 5)
maximum = max(pl1_images[i, j].max() for i in shown_states for j in shown_frames)
for panels, i in zip(axes, shown_states):
    for ax, j, title in zip(panels, shown_frames, ("Reference", "+5 nm sample x", "+1 mrad sample tilt x")):
        artist = ax.imshow(pl1_images[i, j, 108:148, 108:148], origin="lower",
            extent=(-20, 20, -20, 20), cmap="gray", vmin=0, vmax=maximum)
        ax.axhline(0, color="#0072B2", lw=.5, alpha=.6)
        ax.axvline(0, color="#0072B2", lw=.5, alpha=.6)
        ax.set(title=title, xlabel="detector x [pixels]",
               ylabel=f"q = {pl1_q[i]:.3f} rad\ndetector y [pixels]" if j == 0 else "")
fig.colorbar(artist, ax=axes, label="electrons / pixel", shrink=.7)
fig.suptitle("PL1 calibration images · 40×40-pixel crops from 256×256 exposures")
if EXPORT_FIGURES:
    export_ray_figure(fig, OUTPUT / "pl1_image_responses", metadata={
        "report_sha256": hashlib.sha256(pl1_path.read_bytes()).hexdigest(),
        "array_sha256": pl1["arrays_sha256"], "crop_yx": [[108, 148], [108, 148]],
        "scope": "synthetic vacuum input responses; conditional source, not condenser recovery"})
plt.show()


### Physical constraints, a redesigned scan, and held-out images

The nine-coordinate fit releases PL1 length, detector gap, excitation bias/slope, and five upstream magnetic-map coordinates. Training-only QR seeds initialize 64 physical starts. The redesigned relay supplies 162 training images and 27 separate held-out images, retaining the same one-million-electron dose per frame. The table separates recovery error from local standard uncertainty because a fortunate noise realization can give a sub-1% error while several-percent precision remains.

The redesigned fit predicts the held-out responses with χ²≈20.17 for 24 coordinates; after including fitted-parameter uncertainty, predictive χ²≈19.20. A tested competing solution reaches the conditional 0.9 m span bound with Δχ²≈868, but sixteen starts remain unresolved and the finite search does not prove global uniqueness. Two held-out PL1 states also reconstruct the upstream four-row map consistently with propagated analyser uncertainty. Absolute scale still comes from the external position, tilt, and detector calibrations.

In [ ]:
physical_reports = []
for label, name in (("Baseline", "last_lens_physical_fit"),
                    ("Redesigned", "last_lens_redesigned_physical_fit")):
    report = json.loads((GENERATED / f"imaging_recipe/{name}.json").read_text())
    for path, checksum in report["source_hashes"].items():
        if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
            raise ValueError(f"Stale physical fit: {path}")
    for path_key, hash_key in (("image_report_path", "image_report_sha256"), ("arrays_path", "arrays_sha256")):
        if hashlib.sha256((REPO / report[path_key]).read_bytes()).hexdigest() != report[hash_key]:
            raise ValueError(f"Physical-fit input checksum mismatch: {path_key}")
    physical_reports.append((label, report))
rows = ["| Scan | Length error / σ (%) | Gap error / σ (%) | Held-out χ² / coordinates |",
        "|---|---:|---:|---:|"]
for label, report in physical_reports:
    truth, held = report["truth_diagnostic"], report["heldout"]
    error, sigma = truth["length_gap_errors_percent"], truth["length_gap_standard_errors_percent"]
    rows.append(f"| {label} | {error[0]:+.3f} / {sigma[0]:.2f} | "
                f"{error[1]:+.3f} / {sigma[1]:.2f} | {held['chi_square']:.2f} / {held['response_coordinate_count']} |")
display(Markdown("\n".join(rows)))
print("Unresolved physical starts:", {label: r["unresolved_start_count"] for label, r in physical_reports})
print("Every optical fit uses image-derived covariance; no source covariance is supplied to it.")


### Going upstream: all four projector lenses together

The initial IL3–PL1 scan leaves two physical solutions close in training likelihood (Δχ²=2.56). A fresh eight-setting challenge extends IL3 in one connected positive direction; refitting both candidates separates them by Δχ²=108.47, while untouched responses give χ²=32.89/32. Unknown beam shapes alone did not supply that separation. Nine of 22 starts remain unresolved.

The next 72 GPU images scan IL2 at two calibrated downstream analysers. IL2 offsets −0.75, 0, and +0.75 enter training; +0.375 is loaded only after the training winner is fixed. All image fits and selected narrow/wide Gaussian 8→16→32 pixel refinements pass. The controls are synthetic coordinates, not hardware commands.

Noisy relative ray maps initialize IL2 without its generating parameters. The final fit releases all seventeen coordinates and uses the original 224 plus new 48 response coordinates once, without adding the previous calibration posterior as a second prior. Training χ² is 253.52/272 (270.49 at truth); fresh held-out χ² is 13.54/16, or 11.57 after prediction uncertainty.

A further 72 GPU images extend the fit through IL1, using two calibrated downstream analysers and reserving IL1 +0.375 until training finishes. The final twenty-one-coordinate fit uses the previous 272 plus new 48 response coordinates once. Training χ² is 283.68/320 (312.72 at truth), and untouched held-out χ² is 11.97/16, or 11.52 with prediction uncertainty.

Nine of 24 starts converge; the best is interior with rank twenty-one. A tested interior rival is worse by Δχ²=68.29, while fifteen starts remain unresolved. All four projector lenses now have a joint synthetic fit, but IL1 length/gap uncertainties remain 5.50%/3.35%. The accumulated experiment uses 495 probe frames, before condenser or material images; 1% precision, global uniqueness, and the compact full-column budget remain unmet.

In [ ]:
coupled_path = GENERATED / "imaging_recipe/four_projector_recovery.json"
coupled = json.loads(coupled_path.read_text())
for path, checksum in coupled["source_hashes"].items():
    if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
        raise ValueError(f"Stale coupled projector fit: {path}")
rows = ["| Quantity | Fitted error (%) | Local standard uncertainty (%) |", "|---|---:|---:|"]
truth = coupled["truth_diagnostic"]
names = ("PL1 length", "PL1–detector gap", "IL3 length", "IL3–PL1 gap",
         "IL2 length", "IL2–IL3 gap", "IL1 length", "IL1–IL2 gap")
for name, error, sigma in zip(names, truth["length_gap_errors_percent"],
        coupled["information"]["length_gap_standard_errors_percent"]):
    rows.append(f"| {name} | {error:+.2f} | {sigma:.2f} |")
display(Markdown("\n".join(rows)))
print("Converged / total starts:", coupled["converged_start_count"], coupled["start_count"])
print("Calibration posterior counted again as a prior:", coupled["calibration_posterior_added_as_prior"])
print("Global uniqueness proved:", coupled["global_uniqueness_proved"])


In [ ]:
branch_arrays = REPO / coupled["new_arrays_path"]
if hashlib.sha256(branch_arrays.read_bytes()).hexdigest() != coupled["new_arrays_sha256"]:
    raise ValueError("Fresh branch-image checksum mismatch.")
with np.load(branch_arrays, allow_pickle=False) as saved:
    branch_images = saved["noisy_counts"][:, :, 0, 112:144, 112:144]
    branch_controls, branch_training = saved["il1_controls"], saved["training_mask"]
fig, axes = plt.subplots(2, 4, figsize=(11, 6), layout="constrained")
for col, control in enumerate(branch_controls):
    split = "training" if branch_training[col] else "held out"
    for row in range(2):
        artist = axes[row, col].imshow(branch_images[col, row], origin="lower", cmap="gray",
            extent=(-16, 16, -16, 16), vmin=0, vmax=branch_images.max())
        axes[row, col].set(xlabel="detector x [pixel]",
            ylabel=f"Downstream analyser {row+1}\ndetector y [pixel]" if col == 0 else "")
    axes[0, col].set_title(f"IL1 {control:+.3f} · {split}")
fig.colorbar(artist, ax=axes, shrink=.7, label="electrons / pixel")
fig.suptitle("Fresh IL1 experiment · on-axis images · central 32 × 32 crop")
if EXPORT_FIGURES:
    export_ray_figure(fig, OUTPUT / "fresh_il1_images", metadata={
        "report_sha256": hashlib.sha256(coupled_path.read_bytes()).hexdigest(),
        "array_sha256": coupled["new_arrays_sha256"], "crop_pixels_yx": [112, 144, 112, 144]})
plt.show()


### Fewer images, and the external scale floor

For round optics, the reference plus ±x position and ±x tilt images retain enough response information to fit the two-projector model; both detector centroid coordinates remain measured. Refitting the existing images reduces 351 frames to 195 while retaining the tested branch separation. Six orthogonal-input checks add 24 frames, giving a 219-frame candidate. Geometry uncertainty increases by about 46%. The starts came from the earlier nine-input fit, and the orthogonal checks were previously analysed; a cold-start compact acquisition remains to be tested.

Physical length also requires external position, tilt, and detector calibration. The three-projector response has three exact compensating gain families. Independent 0.2% uncertainties in those calibrations, shared across every image, imply a 0.49% common length floor. The value is illustrative until the instrument calibration covariance is measured. The audit includes optics–calibration correlations, so it does not count a shared error again for each exposure.

In [ ]:
checks = {}
for name in ("projector_five_input_ablation", "projector_calibration_audit"):
    report = json.loads((GENERATED / f"imaging_recipe/{name}.json").read_text())
    for path, checksum in report["source_hashes"].items():
        if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
            raise ValueError(f"Stale recipe/calibration audit: {path}")
    checks[name] = report
ablation = checks["projector_five_input_ablation"]
rows = ["| Two-projector inputs | Frames | Tested rival Δχ² |", "|---|---:|---:|"]
for row in ablation["results"]:
    rows.append(f"| {row['label']} | {row['total_frame_count']} | {row['tested_rival_delta_chi_square']:.2f} |")
display(Markdown("\n".join(rows)))
print("With six orthogonal checks:", ablation["y_validation"]["proposed_total_frames"], "frames")
calibration_audit = checks["projector_calibration_audit"]
print("Shared length floor with illustrative calibration (%):",
      round(calibration_audit["common_calibration_length_floor_percent"][0], 3))
print("Data rank / optics plus calibration coordinates:", calibration_audit["data_rank"], "/ 20")
print("Three-projector marginal length/gap σ (%):",
      np.round(calibration_audit["marginalized_length_gap_sigma_percent"], 2))


## 6. The experiment still needs two parts

| Part | Measurements | What it constrains |
|---|---|---|
| Sample → detector | Calibrated sample position/tilt, known layer distances, usable lattice contrast, indexed diffraction | Downstream position and angle responses, scale, rotation and camera length |
| Source → sample | Connected condenser scans; transported astigmatic-beam images at known downstream settings; intermediate-aperture transmission and dose references | Condenser response jointly with unknown entrance-beam moments; conditional recovery below, hardware contract unconfirmed |
| Branch discrimination | Connected one-lens ramps, signed rotation/response, both DAC approach directions | Excitation branches, history dependence and repeatability |

The saved 230-frame dataset covers the first part. Its sources are constructed from declared sample-plane states, so it cannot by itself calibrate CL1, CL3 and C-mini. The historical coupled fit adds stipulated condenser summaries and aperture counts; the current imaging-first investigation replaces its unsupported rotation measurement and tests the actual image information. The current aperture method requires confirmed geometry, repeatable insertion/selection, and verified focus and clearance; calibrated aperture translation is not assumed.

The latest condenser batch uses eight new training settings and three reserved checks, each imaged at two downstream analysers. Stronger CL1/CL3 settings are refocused with C-mini, giving 22 actual Gaussian-mode images. Each image fits its centroid, covariance, and dose using data-derived noise weights. The figure uses one logarithmic count scale so that both narrow and broad beams are visible; negative read-noise values are clipped only for display. These synthetic excitation ranges have not been cleared for hardware.

The capture test now shows that several original states collect only a small fraction of the aperture-out beam; their aperture-in/out ratios cannot supply the assumed total throughput. A separate aperture-imaging collector fits all three nominal aperture images onto the detector, but operates near 2002× specimen magnification and needs a clear vacuum radius of at least 4.33 µm plus alignment margins. The current tiny synthetic hole cannot serve that exposure. [The recipe report](NEOARM_IMAGING_RECIPE.md#collecting-the-condenser-aperture-signal) records the fixed-objective controls and remaining clearance tests.


In [ ]:
from matplotlib.colors import SymLogNorm
beam_path = GENERATED / "imaging_recipe/condenser_extension_images.json"
beam_report = json.loads(beam_path.read_text())
for path, checksum in beam_report["source_hashes"].items():
    if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
        raise ValueError(f"Stale condenser image source: {path}")
beam_arrays = REPO / beam_report["arrays_path"]
if hashlib.sha256(beam_arrays.read_bytes()).hexdigest() != beam_report["arrays_sha256"]:
    raise ValueError("Condenser image checksum mismatch.")
with np.load(beam_arrays, allow_pickle=False) as saved:
    beam_images, beam_training = saved["noisy_beam_images"], saved["training_mask"]
fig, axes = plt.subplots(2, 4, figsize=(11, 6), layout="constrained")
norm = SymLogNorm(linthresh=5, vmin=0, vmax=beam_images.max())
for col, state in enumerate((0, 3, 6, 8)):
    for row in range(2):
        artist = axes[row, col].imshow(np.maximum(beam_images[state, row], 0),
            origin="lower", cmap="gray", norm=norm, extent=(-128, 128, -128, 128))
        axes[row, col].set(xlabel="detector x [pixel]",
            ylabel=f"Analyser {row+1}\ndetector y [pixel]" if col == 0 else "")
    split = "training" if beam_training[state] else "held out"
    axes[0, col].set_title(f"Condenser {state} · {split}")
fig.colorbar(artist, ax=axes, shrink=.7, label="electrons / pixel · logarithmic above 5")
fig.suptitle("Fresh condenser images · propagated Gaussian modes · full detector")
if EXPORT_FIGURES:
    export_ray_figure(fig, OUTPUT / "fresh_condenser_images", metadata={
        "report_sha256": hashlib.sha256(beam_path.read_bytes()).hexdigest(),
        "array_sha256": beam_report["arrays_sha256"], "states": [0, 3, 6, 8]})
plt.show()


### Aperture images supply a physical scale

Aperture-out and three known-radius aperture images are fitted together, releasing beam centroid/covariance, collection magnification, aperture centres, and independent exposure doses. Independent GPU Hermite–Gaussian mode propagation supplies the synthetic reference. A circular edge calibrates length without an aperture translation or a known stigmator location, while its rotational symmetry leaves absolute orientation unmeasured. The recovered spatial covariance is therefore expressed in detector-aligned aperture coordinates.

The statistical magnification uncertainty is around 0.01% in these examples; a shared 0.5% radius uncertainty instead sets a 0.5% scale floor. The images below assume exact aperture conjugacy and a sufficiently large clear specimen region. A measured focus tolerance or a finite-defocus image model is still required for instrument use.

In [ ]:
aperture_path = GENERATED / "imaging_recipe/aperture_image_fit.json"
aperture_fit = json.loads(aperture_path.read_text())
for collection in ("source_hashes", "report_input_hashes"):
    for path, checksum in aperture_fit[collection].items():
        if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
            raise ValueError(f"Stale aperture-image evidence: {path}")
aperture_arrays = REPO / aperture_fit["array_file"]
if hashlib.sha256(aperture_arrays.read_bytes()).hexdigest() != aperture_fit["array_sha256"]:
    raise ValueError("Aperture-image array checksum mismatch.")
family = aperture_fit["records"][0]["family_index"]
with np.load(aperture_arrays, allow_pickle=False) as saved:
    aperture_images = [saved[f"family_{family}_{key}"] for key in
                       ("noisy_images", "predicted_images", "variance_images")]
rows = ["| Beam family | Scale error (%) | Statistical σ (%) | Image-fit accepted |", "|---|---:|---:|---|"]
for row in aperture_fit["records"]:
    rows.append(f"| {row['family_index']} | {100*row['magnification_relative_error']:+.4f} | "
                f"{100*row['magnification_relative_standard_error']:.4f} | {row['accepted']} |")
display(Markdown("\n".join(rows)))
print("Masked finite-defocus propagation certified:", aperture_fit["finite_B_masked_wave_error_certified"])


In [ ]:
observed, predicted, variance = aperture_images
fig, axes = plt.subplots(2, 4, figsize=(12, 6), layout="constrained")
for col, title in enumerate(("Aperture out", "10 µm radius", "20 µm radius", "30 µm radius")):
    counts_artist = axes[0, col].imshow(observed[col], origin="lower", cmap="gray",
        extent=(-7.04, 7.04, -7.04, 7.04), vmin=0, vmax=observed.max())
    difference = (observed[col]-predicted[col])/np.sqrt(variance[col])
    residual_artist = axes[1, col].imshow(difference, origin="lower", cmap="RdBu_r",
        extent=(-7.04, 7.04, -7.04, 7.04), vmin=-4, vmax=4)
    axes[0, col].set_title(title)
    for row in (0, 1):
        axes[row, col].set(xlabel="detector x [mm]", ylabel="detector y [mm]" if col == 0 else "")
fig.colorbar(counts_artist, ax=axes[0], shrink=.8, label="electrons / pixel")
fig.colorbar(residual_artist, ax=axes[1], shrink=.8, label="data − fit [noise σ]")
fig.suptitle("Aperture-image calibration · exact-conjugate synthetic state")
if EXPORT_FIGURES:
    export_ray_figure(fig, OUTPUT / "aperture_image_fits", metadata={
        "report_sha256": hashlib.sha256(aperture_path.read_bytes()).hexdigest(),
        "array_sha256": aperture_fit["array_sha256"], "family": family,
        "scope": "known-radius spatial metrology at exact conjugacy; not condenser-lens recovery"})
plt.show()


### Condenser: a stationary conditional fit

The fit combines 102 training beam/aperture coordinates with one common 0.5% aperture-radius prior. Six lens-centre/length coordinates form the outer fit; the six excitation-law coordinates, nine visible source-covariance coordinates, aperture-frame rotation, and radius error are fitted internally at each geometry trial. All twelve lens parameters remain free in the combined likelihood, and the tenth source coordinate retains its quantum-admissible interval. No independent ellipse-angle measurement or calibrated upstream kick is supplied.

The first nested fit converges in seventeen outer steps, with data χ²=92.21/102 and original-coordinate normalized gradient 8.85×10⁻⁹. CL1/CL3/C-mini length errors are +1.36%/+1.16%/+0.78%, with conditional local uncertainties of 1.04%/0.88%/0.35%. This direct-profile benchmark continues from a training fit near the generating optical seed, while its offset starts fail. The later moment-initialized cold route below reaches the same minimum from a displaced geometry. The source is highly chirped, but an admissible Gaussian representative exists.

The training solution is frozen before the fresh held-out observations are loaded. Their point predictions miss the image-statistical errors (raw χ²=166.54/18), while the local fitted-parameter covariance gives predictive χ²=22.41/18. Beam-diameter errors remain below 0.468%, within the plan's 2% tolerance. This is a local uncertainty check, not demonstrated confidence coverage.

A nonlinear profile moves the CL1 centre by +2.667 mm and refits all other parameters, reaching a stationary solution only Δχ²=0.8933 above the best fit. Thus distinct geometries remain compatible with these measurements. The negative one-sigma target has a feasible candidate at Δχ²=0.9513 but unresolved final stationarity; both two-sigma targets remain unresolved. Failed profile searches are not exclusions.

At the fitted source and optics, all six held-out maps also pass independent Gaussian-mode and pixel refinements. The worst detector discrepancy is 0.000997 noise σ RMS and joint observable bias is 0.00123σ, so Gaussian discretization does not explain the point-prediction residuals.

Objective/downstream calibration, aperture position, clear paths, and unchanged upstream astigmatic preparation remain conditional. With an unknown internal stigmator and all ten source moments free, independent CL1/CL3/C-mini scans retain ideal local optical rank; holding upstream lenses fixed leaves exact equivalent families. The original composed-matrix guard failure is now bypassed by a validated physical-factor propagation, without weakening the guard. Three selected states pass modal, transport, and pixel refinements. Full acquisition remains paused because exact uniform-body ray transport predicts changes far above image noise for some settings, including settings in the older conditional condenser benchmark. These are classical moment-sensitivity tests, not quantum wave calculations or measured NEOARM aberrations. This internal-stigmator model still has no accepted image-recovery demonstration; see the [numerical validation](../generated/neoarm_inversion/imaging_recipe/condenser_factorized_validation.json) and [nonparaxial benchmark](../generated/neoarm_inversion/imaging_recipe/condenser_nonparaxial_benchmark.json).

In [ ]:
reports = []
for name in ("condenser_guarded_recovery", "condenser_stationary_validation", "condenser_fitted_gaussian_validation"):
    report = json.loads((GENERATED / f"imaging_recipe/{name}.json").read_text())
    for path, checksum in report["source_hashes"].items():
        if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
            raise ValueError(f"Stale condenser checkpoint source: {path}")
    reports.append(report)
condenser, validation, fitted_gaussians = reports
assert validation["stationary_conditional_training_fit"]
assert fitted_gaussians["all_six_fitted_heldout_images_accepted"]
assert validation["frozen_selection"]["no_heldout_or_generating_values_used"]
held = validation["fresh_heldout"]
rows = ["| Evaluation | Training χ² / 102 | Fresh held-out χ² / 18 | Predictive χ² / 18 |", "|---|---:|---:|---:|",
    f"| Synthetic truth | {condenser['truth_diagnostic']['training_chi_square']:.2f} | — | — |",
    f"| Stationary conditional fit | {validation['training_data_chi_square']:.2f} | {held['chi_square']:.2f} | {held['predictive_chi_square']:.2f} |"]
display(Markdown("\n".join(rows)))
print("Instrument-ready condenser recovery:", validation["hardware_ready"])
print("Normalized gradient:", validation["original_normalized_gradient"])
print("Radius prior is included once in the fit, excluded from this data-only χ² table.")


### A specimen image in the same shared-objective column

This new 40k exposure uses the same objective field and downstream column as the condenser calibration. The objective controls are unchanged; a third projector setting makes the specimen conjugate to the detector. The same cubic atlas is displaced by 3 nm and −4 nm, with a prescribed 110 nm canonical sample-plane probe. It is not illumination transported from the recovered condenser source.

Three Gaussian grids contain 2.63, 5.26, and 10.50 million beamlets. Their pixel-integrated intensities pass an independent cubic conjugate-image reference, and all seven image parameters are fitted jointly. The two successive grid refinements change fitted parameters by less than 0.1 local standard errors. Noise weights come from fitted counts; measured endpoint calibration supplies the starting scale/rotation and a broad rotation search bound, not a likelihood prior. The fixed crop covers the full declared fitting box.

The numerical checks admit terminal detector intensity at this optical state. They do not certify arbitrary defocus, nonlinear transport, the real specimen contrast, or complete lens recovery. The images below show a full 256×256 detector, with no residual smoothing.

In [ ]:
shared_material = json.loads((GENERATED / "imaging_recipe/shared_material_images.json").read_text())
for path, checksum in shared_material["source_hashes"].items():
    assert hashlib.sha256((REPO / path).read_bytes()).hexdigest() == checksum, path
assert shared_material["all_passed"]
material_path = REPO / shared_material["arrays_path"]
assert hashlib.sha256(material_path.read_bytes()).hexdigest() == shared_material["arrays_sha256"]
with np.load(material_path, allow_pickle=False) as archive:
    material_images = {name: archive[name].copy() for name in
        ("noisy_counts_yx", "fitted_counts_yx", "fitted_variance_yx")}
material_fit = shared_material["noisy_fit"]
display(Markdown("| Shared-column specimen fit | Result |\n|---|---:|\n"
    f"| Magnification error | {100*material_fit['magnification_relative_error']:.4f}% |\n"
    f"| Rotation error | {material_fit['rotation_error_degrees']:.4f}° |\n"
    f"| Image residual RMS | {material_fit['data_rms_sigma']:.4f} σ |\n"
    f"| Finest Gaussian/reference difference | {shared_material['refinements'][-1]['gaussian_reference_joint_sigma']:.5f} whitened-L2 σ |"))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.9), layout="constrained")
extent = np.array([-128, 128, -128, 128])*55e-3
vmax = float(np.percentile(material_images["fitted_counts_yx"], 99.9))
for ax, key, title in zip(axes[:2], ("noisy_counts_yx", "fitted_counts_yx"),
        ("GPU Gaussian exposure", "Seven-parameter cubic fit")):
    artist = ax.imshow(np.maximum(material_images[key], 0), origin="lower", extent=extent,
        cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_title(title)
    ax.plot([-5.8, -3.8], [-5.8, -5.8], color="#E69F00", lw=3)
    ax.text(-5.8, -5.1, "50 nm · model scale", color="#E69F00", fontsize=8)
residual = (material_images["noisy_counts_yx"]-material_images["fitted_counts_yx"])
residual /= np.sqrt(material_images["fitted_variance_yx"])
residual_artist = axes[2].imshow(residual, origin="lower", extent=extent,
    cmap="RdBu_r", vmin=-4, vmax=4, interpolation="nearest")
axes[2].set_title("Data − fit / fitted noise σ")
for ax in axes: ax.set(xlabel="detector x [mm]", xticks=(-6, 0, 6), yticks=(-6, 0, 6))
axes[0].set_ylabel("detector y [mm]")
fig.colorbar(artist, ax=axes[:2], shrink=.75, label="electrons / pixel")
fig.colorbar(residual_artist, ax=axes[2], shrink=.75, label="σ")
fig.suptitle("40k specimen imaging in the condenser's shared-objective column")
if EXPORT_FIGURES:
    export_ray_figure(fig, OUTPUT / "shared_material_40k", metadata={
        "title": "Same-column Gaussian specimen exposure and image fit",
        "array_sha256": shared_material["arrays_sha256"], "beam_count": 10504081,
        "detector_pixel_m": 55e-6, "specimen_scale_bar_m": 50e-9,
        "negative_read_noise_clipped_only_for_display": True})
plt.show()


### Downstream calibration measured in the same column

The earlier four-projector experiment and the condenser experiment use different objective fields. Matching every controlled-lens excitation still leaves response differences up to 1.23 pixels, so their full calibration posteriors cannot be combined. The new calibration uses the condenser column itself, at two settings near 40k. Eighteen GPU Gaussian images measure the signed responses to 5 nm position and 1 mrad tilt inputs; eight further images provide independent diagonal-input checks.

The joint fit releases all 23 condenser/source/aperture coordinates and six canonical downstream position-row coordinates, using 102 condenser observations, 16 new calibration responses, and one common radius prior. A cold pipeline now reaches this solution without a saved fitted geometry: a moment-based source projection initializes the original physical condenser likelihood, then the six endpoint coordinates are initialized from measured responses and all 29 parameters are released together. One of three prescribed starts succeeds, beginning with log-length offsets −0.08/+0.06/−0.10 and centre offsets −1/+0.3/−0.15 mm. The other starts remain unresolved.

The objective prefield and sample magnetic shear remain fixed, and the six endpoint coordinates describe measured maps rather than individual projector lenses. A five-input ablation suggests ten calibration images plus eight orthogonal checks may suffice, with 26% higher C-mini position uncertainty. That smaller experiment reuses inspected data and still needs independent acquisition.

The fit converges in four outer updates, and an independent audit finds a positive full local Hessian and likelihood consistency at synthetic truth. Its reserved calibration checks agree with image noise; the previously inspected condenser checks still need parameter-prediction uncertainty to explain their residuals. At the same fitted parameters, including endpoint-map uncertainty increases the C-mini position uncertainty by 79%. The figure shows central detector crops on one count scale, with fitted centroids marked. The full saved images are 256×256 pixels.

In [ ]:
endpoint_reports = []
endpoint_names = ("condenser_endpoint_calibration", "condenser_endpoint_recovery",
    "condenser_endpoint_validation", "condenser_cold_joint_endpoint", "condenser_endpoint_ablation")
for name in endpoint_names:
    report = json.loads((GENERATED / f"imaging_recipe/{name}.json").read_text())
    for path, checksum in report["source_hashes"].items():
        if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
            raise ValueError(f"Stale same-column calibration: {path}")
    endpoint_reports.append(report)
endpoint_images, endpoint_fit, endpoint_validation, cold_joint, endpoint_ablation = endpoint_reports
assert endpoint_images["all_images_accepted"] and endpoint_fit["accepted_stationary_fit"]
assert endpoint_validation["positive_local_minimum"]
assert cold_joint["accepted_stationary_fit"] and endpoint_ablation["accepted_stationary_fit"]
endpoint_archive = REPO / endpoint_images["arrays_path"]
assert hashlib.sha256(endpoint_archive.read_bytes()).hexdigest() == endpoint_images["arrays_sha256"]
with np.load(endpoint_archive, allow_pickle=False) as archive:
    calibration_counts = archive["noisy_counts"].copy()
    calibration_centroids = archive["image_parameters"][..., :2].copy()
held = endpoint_fit["new_heldout_calibration"]
display(Markdown("| Same-column joint fit | χ² | Coordinates |\n|---|---:|---:|\n"
    f"| Training data | {endpoint_fit['training_data_chi_square']:.2f} | 118 |\n"
    f"| Reserved calibration inputs | {held['raw_chi_square']:.2f} | 8 |\n"
    f"| Reserved inputs, local prediction covariance | {held['local_predictive_chi_square']:.2f} | 8 |"))
print("Conditional lens-length uncertainties [%]:", np.round(endpoint_fit["condenser_length_standard_errors_percent"], 3))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6), sharex=True, sharey=True, layout="constrained")
norm = SymLogNorm(linthresh=5., vmin=0., vmax=float(calibration_counts.max()))
for row in range(2):
    for col, (frame, title) in enumerate(((0, "Reference"), (1, "Position +5 nm"), (5, "Tilt +1 mrad"))):
        ax = axes[row, col]
        artist = ax.imshow(np.maximum(calibration_counts[row, frame], 0), origin="lower",
            extent=(-128, 128, -128, 128), cmap="magma", norm=norm, interpolation="nearest")
        ax.plot(*calibration_centroids[row, frame], marker="+", color="#56B4E9", ms=8, mew=1)
        ax.set(xlim=(-10, 10), ylim=(-10, 10), xticks=(-10, 0, 10), yticks=(-10, 0, 10))
        if row == 0: ax.set_title(title)
        if col == 0: ax.set_ylabel(f"Analyser {row+1} · y [pixels]")
        if row == 1: ax.set_xlabel("detector x [pixels]")
fig.colorbar(artist, ax=axes, shrink=.8, label="electrons / pixel · logarithmic above 5")
fig.suptitle("Same-column calibration · two near-40k states\nCentral 20×20-pixel crops · 55 µm detector pitch")
if EXPORT_FIGURES:
    export_ray_figure(fig, OUTPUT / "condenser_endpoint_calibration", metadata={
        "title": "Same-column signed-input calibration images",
        "array_sha256": endpoint_images["arrays_sha256"], "frames": [0, 1, 5],
        "detector_pixel_m": 55e-6, "crop_pixels": [-10, 10, -10, 10],
        "negative_read_noise_clipped_only_for_display": True})
plt.show()


### What is identifiable?

The joint information test releases the optical parameters, source covariance, alignment, and calibration uncertainties together. A wide same-sign PL1 sweep helps considerably, but several geometry directions remain weak. The source reference plane retains an exact covariance-transport ambiguity: changing that reference and transporting the unknown source covariance gives the same measurements. It cannot be reported as a uniquely recovered physical source position.

The new model joins objective prefield and postfield through one field parameterised by strength, peak, width, and asymmetry, converting physical specimen tilts to canonical ray coordinates. Selected-frame field and pixel refinements pass. Continuing the assumed tails across the full column changes rotation by about 5.25° with all controls and profile amplitude fixed; that is sensitivity to an unmeasured field hypothesis, not proof of the true far field. The earlier scale survey retains independent objective halves; the new 40k material image uses the shared finite-support objective, while Section 7 traces the overlapping-field hypothesis.

The physical PL1 fits in Section 5 enforce the magnetic-symplectic constraint and improve the experiment through relay selection. Their length/gap uncertainties are approximately 9%/8.5% for the baseline and 3.4%/3.1% for the redesigned scan. Distinct excitation starts remain unresolved, and the conditional geometry bound still participates in rejecting tested competitors. The [recipe report](NEOARM_IMAGING_RECIPE.md#a-smaller-downstream-experiment-last-lens-tomography) distinguishes this evidence from the earlier volume-only relaxation.

The older 64-start material-image fit failed. Its likelihood is inconsistent even at synthetic truth; increasing its start count cannot repair incorrectly registered specimen peaks or clipped-beam centroid predictions. The new route reconstructs finite-detector Gaussian moments, estimates signed section responses, initializes excitation before geometry, and then minimizes the joint likelihood in sensitivity-scaled coordinates with all physical bounds retained.

The adjacent-branch challenge tests explicit one-lens phase hypotheses. A failed branch optimization remains unresolved, and endpoint approach metadata are not measured continuation. Connected scans and independent geometry still anchor the instrument branch.

Hard-edge solenoids are effective finite-field models with symplectic transport and Larmor rotation. Smooth fields require held-out evidence for their additional parameters.

## 7. Ray diagrams with honest physical scales

The figures below use the approximate measured diagram geometry and the full shared objective field, including its overlap with all seven controlled lenses. The previous 40k/80k controls are unchanged, giving approximately 39.73k and 79.52k in this field hypothesis. The overview uses true z; condenser, objective, and projector panels retain linear physical axes. Blue is the height basis, vermillion the angle basis, and pale blue the Gaussian envelope. The objective glyph marks its field FWHM, not a polepiece or a physical field boundary.

A circular canonical Gaussian has a 60 nm sample-plane reference radius and a flat radial wavefront, with basis scales $w_0$ and $\lambda/(\pi w_0)$. Inside a magnetic lens that reference plane need not be a radius minimum; the annotated waists are separately located minima of $w(z)$. Back-propagating the reference rays defines the source-plane basis, and the envelope $w=\sqrt{x_h^2+x_\alpha^2}$ agrees with independent covariance propagation. The second figure uses mechanical 5 nm / 1 mrad sample inputs. Sample immersion gives the position ray an additional orthogonal response, saved alongside the plotted co-rotating x projection; these calibrated rays have no Gaussian-envelope interpretation.

A/B zeros and waist minima are refined in the continuous numerical field, and detector ray heights are checked against JAX single-ray Jacobians. The Larmor strip integrates the summed signed field through source and detector. SVG, PDF, and 300-dpi PNG outputs include source hashes, numerical checks, and common-scale IL3 wobble comparisons. These are predictions of the specified field, not fitted full-column geometry.


In [ ]:
import hashlib
RAY_MAGNIFICATION = MAGNIFICATION if MAGNIFICATION in (40_000, 80_000) else 40_000
RAY_OUTPUT = GENERATED / "imaging_recipe/review/shared_field"
ray_report = json.loads((RAY_OUTPUT / "ray_review.json").read_text())
for path, checksum in ray_report["source_hashes"].items():
    if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
        raise ValueError(f"Stale shared-field ray source: {path}")
if hashlib.sha256((REPO / ray_report["array_file"]).read_bytes()).hexdigest() != ray_report["array_sha256"]:
    raise ValueError("Shared-field ray-array checksum mismatch.")

def show_shared_rays(name):
    record = ray_report["figures"][name]
    if record["label_collisions"]:
        raise ValueError("Ray-figure label overlap audit failed.")
    for path, checksum in record["file_hashes"].items():
        if hashlib.sha256((REPO / path).read_bytes()).hexdigest() != checksum:
            raise ValueError(f"Changed ray-figure file: {path}")
    stem = REPO / record["stem"]
    display(Image(filename=str(stem.with_suffix(".png")), width=1500))

show_shared_rays(f"nominal_{RAY_MAGNIFICATION//1000}k")


In [ ]:
show_shared_rays("calibrated_40k")
show_shared_rays("IL3_comparison")
print("Approximate-geometry synthetic controls:", ray_report["controls"]["nominal_40k"])
print("These controls are not a hardware command sequence.")


## 8. What remains before using this on the NEOARM?

1. **Validate the objective model.** Test field tails and overlapping lens fields; retain the immersed specimen's mechanical/canonical conversion. Numerical convergence of a truncated field is not evidence that its physical extent is correct.
2. **Improve and simplify the four-projector fit.** All four lenses now pass conditional noisy recovery and fresh held-out checks, with IL1 still weak. Retain unresolved branches, include measured calibration covariance, and improve precision through measurement design or independently calibrated geometry. The reduced five-input recipe needs its own training-only initialization and a full-projector test.
3. **Fit the measured-geometry images.** Combine the pixel-integrated condenser model and cubic specimen estimator, include calibration uncertainty, and demonstrate blind coupled recovery and refinement of every identifiable optical direction. Bind real control units and safe ranges; stigmator location and aperture translation capability remain unconfirmed.
4. **Validate real held-out exposures.** Fit both measurement blocks and test optical predictions on states excluded from fitting before claiming a calibrated NEOARM model.

The new probe likelihood uses analytic noise covariance without bootstrap regeneration.

### Implementation

- [neoarm_measured_initialization.py](neoarm_measured_initialization.py): first-branch diagram-geometry imaging states and bounded projection changes.
- [neoarm_finite_pixel_metrology.py](neoarm_finite_pixel_metrology.py): GPU pixel-integrated specimen images and seven-parameter image fits.
- [neoarm_shared_objective.py](neoarm_shared_objective.py): shared-field column and mechanical-input ray Jacobians.
- [neoarm_shared_objective_audit.py](neoarm_shared_objective_audit.py): coupled optical/source/calibration information and selected-frame numerical refinement.
- [neoarm_last_lens_tomography.py](neoarm_last_lens_tomography.py): last-lens reconstruction with unknown upstream optics; conditional response precision.
- [gaussian_image_likelihood.py](../../src/temgym_core/gaussian_image_likelihood.py): pixel-area-integrated Gaussian images, six-parameter fits, and exact local image-information factors.
- [neoarm_recovery.py](neoarm_recovery.py): truth checks, initialization, bounded coupled fitting, physical recovery diagnostics, and branch challenges.
- [neoarm_probe_likelihood.py](neoarm_probe_likelihood.py): unclipped probe moments, analytic covariance, and image-derived Gaussian initialization.
- [neoarm_workflow.py](neoarm_workflow.py): checked acquisitions, specimen images, GPU reproduction, and ray tracing.
- [multiscale_sample.py](../../src/temgym_core/multiscale_sample.py): physical atlas and cubic action fields.
- [phase_branch_continuation.py](../../src/temgym_core/phase_branch_continuation.py): connected-scan contracts.
- [Research archive](archive/neoarm_full_column_research.ipynb): preserved historical orchestration; not executed here.